In [ ]:
%%capture
%pip install -q transformers librosa soundfile matplotlib pandas numpy tqdm pesq pystoi jiwer scipy scikit-learn scikit-image

In [ ]:
"""Reproducible evaluation for Dynamic PGD + U-Shape voice protection.

The module intentionally reports SQUIM-MOS as SQUIM-MOS.  It never labels a
proxy or a hand-written formula as NISQA. The fixed routing target is kept
inside PGD, while reported ASV endpoints use a speaker-disjoint EER threshold.
"""

from __future__ import annotations

import json
import math
import os
import warnings

os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
warnings.filterwarnings('ignore')
import platform
import random
import re
from dataclasses import asdict, dataclass, field
from itertools import combinations
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import librosa
import librosa.display
import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchaudio
import soundfile as sf
from matplotlib import pyplot as plt
from pesq import pesq
from pystoi import stoi
from scipy import stats
from scipy.fft import dct
from sklearn.metrics import roc_curve
from skimage.metrics import structural_similarity
from tqdm.auto import tqdm
from transformers import (
    HubertModel,
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    WavLMForXVector,
)
from transformers.utils import logging as transformers_logging
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()

# MODE EKSEKUSI: True = hitung ulang statistik dari hasil lama tanpa dataset audio.
POSTPROCESS_ONLY = False
# True = hitung ulang ASV EER dengan protokol centroid 2-enrollment lalu
# perbarui endpoint/statistik hasil lama; PGD dan metrik audio tidak dijalankan.
RECALIBRATE_ASV_ONLY = True
# Kaggle biasanya sudah mengekstrak ZIP input. None = cari hasil LibriTTS otomatis di /kaggle/input.
# Boleh juga diisi path folder hasil ekstraksi atau path file ZIP.
POSTPROCESS_SOURCE = None  # contoh folder: '/kaggle/input/hasil-revisi'

@dataclass
class ExperimentConfig:
    # Dataset protocol: speakers in calibration and test are disjoint.
    dataset_name: str = "LibriTTS"
    dataset_glob: str = "./datasets/libri-tts-train-clean-360-part-2/**/*.wav"
    output_dir: str = "./hasil_revisi_libritts"
    sample_rate: int = 16_000
    segment_seconds: float = 4.0
    minimum_seconds: float = 1.0
    calibration_speakers: int = 20
    calibration_utterances_per_speaker: int = 5
    test_speakers: int = 40
    enrollment_utterances_per_speaker: int = 2
    test_utterances_per_speaker: int = 8
    expected_unique_utterances: int = 500
    selection_seed: int = 2026
    normalize_input_level: bool = True
    input_target_rms_dbfs: float = -26.0
    input_peak_limit: float = 0.95

    # Repeated randomized PGD protocol.
    run_seeds: Tuple[int, ...] = (42, 43, 44)
    num_iterations: int = 1000
    step_size: float = 1e-3
    epsilon_max: float = 0.12
    baseline_noise: float = 0.02
    checkpoint_every: int = 10

    # Target models and the operational definition of F.
    wavlm_checkpoint: str = "microsoft/wavlm-base-plus-sv"
    hubert_checkpoint: str = "facebook/hubert-base-ls960"
    semantic_layer: int = 2
    asr_checkpoint: str = "facebook/wav2vec2-base-960h"
    run_asr_wer: bool = True

    # STFT is fully specified; frequency regions are expressed in Hz.
    n_fft: int = 512
    win_length: int = 512
    hop_length: int = 160
    stft_center: bool = True
    stft_normalized: bool = False
    # Exact Hz equivalents of STFT bins 80, 45, and 120 at 16 kHz/512.
    phase1_high_cut_hz: float = 2500.0
    ushape_low_end_hz: float = 1406.25
    ushape_mid_end_hz: float = 3750.0
    phase1_high_scale: float = 0.05
    ushape_mid_scale: float = 0.01
    ushape_high_scale: float = 0.15
    high_frequency_start_hz: float = 3750.0

    # Equation (9) uses Mel bins [0:45] exactly.
    n_mels: int = 128
    mel_fmin: float = 0.0
    mel_fmax: float = 8000.0
    mel_scale: str = "htk"
    mel_norm: Optional[str] = None
    mel_power: float = 2.0
    formant_mel_bins: int = 45

    # Static PGD holds State-A weights fixed. Dynamic variants may switch states.
    static_weights: Dict[str, float] = field(default_factory=lambda: {
        "identity": 150.0, "formant": 30.0, "hubert_identity": 20.0, "semantic": 5.0
    })
    phase1_weights: Dict[str, float] = field(default_factory=lambda: {
        "identity": 150.0, "formant": 30.0, "hubert_identity": 20.0, "semantic": 5.0
    })
    phase2_weights: Dict[str, float] = field(default_factory=lambda: {
        "identity": 15.0, "formant": 300.0, "hubert_identity": 20.0, "semantic": 20.0
    })

    # Fixed operating points narrated in Sections III-C and IV-A.
    routing_threshold: float = 0.24
    asv_evaluation_threshold: float = 0.25
    quality_calibration_snr_db: float = 10.0
    quality_metric_name: str = "SQUIM_MOS"
    save_audio_examples: bool = True
    strict_paper_method: bool = True

    def validate(self) -> None:
        assert self.sample_rate == 16_000, "PESQ wide-band evaluation requires 16 kHz here."
        assert self.mel_fmax <= self.sample_rate / 2
        assert 0 < self.ushape_low_end_hz < self.ushape_mid_end_hz <= self.sample_rate / 2
        assert 0 < self.formant_mel_bins <= self.n_mels
        assert self.win_length <= self.n_fft
        assert self.input_target_rms_dbfs < 0 and 0 < self.input_peak_limit <= 1.0
        assert len(self.run_seeds) >= 2, "Use at least two repeated runs; three or more are recommended."
        expected = (
            self.calibration_speakers * self.calibration_utterances_per_speaker
            + self.test_speakers * (self.enrollment_utterances_per_speaker + self.test_utterances_per_speaker)
        )
        assert expected == self.expected_unique_utterances, f"Configured selection is {expected}, not {self.expected_unique_utterances} utterances."
        if self.strict_paper_method:
            assert self.num_iterations == 1000 and self.step_size == 1e-3
            assert self.epsilon_max == 0.12 and self.baseline_noise == 0.02
            assert self.routing_threshold == 0.24 and self.asv_evaluation_threshold == 0.25
            assert self.static_weights == {"identity": 150.0, "formant": 30.0, "hubert_identity": 20.0, "semantic": 5.0}
            assert self.phase1_weights == self.static_weights
            assert self.phase2_weights == {"identity": 15.0, "formant": 300.0, "hubert_identity": 20.0, "semantic": 20.0}
            assert self.phase1_high_scale == 0.05 and self.ushape_mid_scale == 0.01 and self.ushape_high_scale == 0.15
            resolution = self.sample_rate / self.n_fft
            assert self.phase1_high_cut_hz / resolution == 80
            assert self.ushape_low_end_hz / resolution == 45
            assert self.ushape_mid_end_hz / resolution == 120
            assert self.formant_mel_bins == 45


METHODS = (
    "Undefended",
    "Random Noise",
    "Static PGD",
    "Dynamic PGD (No U-Shape)",
    "High-Frequency PGD (3.75-8 kHz)",
    "Proposed (Dynamic + U-Shape)",
)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def speaker_id(path: str, cfg: ExperimentConfig) -> str:
    """Dataset-specific speaker ID without changing the experiment method."""
    audio = Path(path)
    if cfg.dataset_name.lower() == "vctk":
        # VCTK: <root>/wav48[_silence_trimmed]/pXXX/<utterance>.wav
        return audio.parent.name
    # LibriTTS: <root>/<speaker>/<chapter>/<utterance>.wav
    return audio.parent.parent.name


def utterance_id(path: str) -> str:
    return Path(path).stem


def source_utterance_id(path: str, cfg: ExperimentConfig) -> str:
    stem = utterance_id(path)
    return re.sub(r"_mic[12]$", "", stem) if cfg.dataset_name.lower() == "vctk" else stem


def audio_metadata(path: str) -> Dict[str, float]:
    info = sf.info(path)
    return {
        "duration_s": float(info.frames) / float(info.samplerate),
        "original_sample_rate_hz": int(info.samplerate),
        "original_channels": int(info.channels),
    }


def audio_duration(path: str) -> float:
    return audio_metadata(path)["duration_s"]




In [ ]:
def build_speaker_disjoint_split(cfg: ExperimentConfig) -> pd.DataFrame:
    """Select complete utterances in the configured duration interval."""
    import glob
    files = sorted(glob.glob(cfg.dataset_glob, recursive=True))
    if not files:
        raise FileNotFoundError(f"No WAV files matched: {cfg.dataset_glob}")

    rows = []
    for path in files:
        metadata = audio_metadata(path)
        duration = metadata["duration_s"]
        if cfg.minimum_seconds <= duration <= cfg.segment_seconds:
            rows.append({
                "path": path, "speaker_id": speaker_id(path, cfg),
                "utterance_id": utterance_id(path), "source_utterance_id": source_utterance_id(path, cfg),
                **metadata,
            })
    eligible = pd.DataFrame(rows)
    if eligible.empty:
        raise RuntimeError("No utterance satisfies the configured duration interval.")
    # Avoid treating VCTK mic1/mic2 copies of the same spoken sentence as
    # independent enrollment/probe samples. Sorting deterministically keeps mic1.
    eligible = eligible.sort_values("path").drop_duplicates(
        subset=["speaker_id", "source_utterance_id"], keep="first"
    ).reset_index(drop=True)

    required_test = cfg.enrollment_utterances_per_speaker + cfg.test_utterances_per_speaker
    counts = eligible.groupby("speaker_id").size()
    calib_candidates = set(counts[counts >= cfg.calibration_utterances_per_speaker].index)
    test_candidates = set(counts[counts >= required_test].index)
    candidates = sorted(calib_candidates | test_candidates)
    rng = random.Random(cfg.selection_seed)
    rng.shuffle(candidates)

    calibration_speakers = [s for s in candidates if s in calib_candidates][: cfg.calibration_speakers]
    test_speakers = [s for s in candidates if s not in calibration_speakers and s in test_candidates][: cfg.test_speakers]
    if len(calibration_speakers) < cfg.calibration_speakers or len(test_speakers) < cfg.test_speakers:
        raise RuntimeError(
            f"Insufficient speakers after filtering: calibration={len(calibration_speakers)}, "
            f"test={len(test_speakers)}. Reduce requested counts or duration restrictions."
        )

    selected = []
    for spk in calibration_speakers:
        group = eligible[eligible.speaker_id == spk].sample(
            n=cfg.calibration_utterances_per_speaker,
            random_state=cfg.selection_seed + sum(map(ord, spk)),
        ).sort_values("utterance_id")
        for row in group.to_dict("records"):
            selected.append({**row, "split": "calibration", "role": "calibration"})
    for spk in test_speakers:
        group = eligible[eligible.speaker_id == spk].sample(
            n=required_test,
            random_state=cfg.selection_seed + sum(map(ord, spk)),
        ).sort_values("utterance_id")
        for j, row in enumerate(group.to_dict("records")):
            role = "enrollment" if j < cfg.enrollment_utterances_per_speaker else "probe"
            selected.append({**row, "split": "test", "role": role})
    result = pd.DataFrame(selected)
    if len(result) != cfg.expected_unique_utterances:
        raise RuntimeError(f"Selection produced {len(result)} instead of exactly {cfg.expected_unique_utterances} utterances.")
    return result


def canonicalize_audio_level(wav: torch.Tensor, cfg: ExperimentConfig) -> torch.Tensor:
    """Dataset-independent level normalization; it does not alter pitch or timing."""
    wav = wav - wav.mean(dim=-1, keepdim=True)
    if not cfg.normalize_input_level:
        return wav.clamp(-1.0, 1.0)
    target_rms = 10.0 ** (cfg.input_target_rms_dbfs / 20.0)
    current_rms = torch.sqrt(torch.mean(wav.square(), dim=-1, keepdim=True) + 1e-12)
    wav = wav * (target_rms / current_rms)
    peak = wav.abs().amax(dim=-1, keepdim=True).clamp_min(1e-12)
    peak_scale = torch.minimum(torch.ones_like(peak), torch.full_like(peak, cfg.input_peak_limit) / peak)
    return (wav * peak_scale).clamp(-1.0, 1.0)


def load_audio(path: str, cfg: ExperimentConfig, device: torch.device) -> torch.Tensor:
    waveform_np, sr = sf.read(path, dtype='float32', always_2d=True)
    waveform_np = waveform_np.mean(axis=1)
    wav = torch.from_numpy(waveform_np).unsqueeze(0)
    if sr != cfg.sample_rate:
        wav = torchaudio.functional.resample(wav, sr, cfg.sample_rate)
    maximum_length = round(cfg.segment_seconds * cfg.sample_rate)
    if wav.shape[-1] > maximum_length:
        raise ValueError(f"Selection protocol forbids truncation, but {path} is too long.")
    # Keep the original duration. Padding created artificial noisy tails and
    # biased PESQ/STOI when the perturbation was optimized on padded silence.
    return canonicalize_audio_level(wav, cfg).to(device)


def normalize_waveform(x: torch.Tensor) -> torch.Tensor:
    return (x - x.mean(dim=-1, keepdim=True)) / torch.sqrt(x.var(dim=-1, keepdim=True, unbiased=False) + 1e-7)


class RepresentationModels:
    """R=WavLM-SV embedding, H=HuBERT last layer, F=HuBERT hidden layer k."""

    def __init__(self, cfg: ExperimentConfig, device: torch.device):
        self.cfg = cfg
        self.device = device
        self.wavlm = WavLMForXVector.from_pretrained(cfg.wavlm_checkpoint).to(device).eval()
        self.hubert = HubertModel.from_pretrained(cfg.hubert_checkpoint).to(device).eval()
        for model in (self.wavlm, self.hubert):
            for parameter in model.parameters():
                parameter.requires_grad_(False)

    def extract(self, waveform: torch.Tensor, with_grad: bool = False) -> Dict[str, torch.Tensor]:
        context = torch.enable_grad() if with_grad else torch.no_grad()
        with context:
            x = normalize_waveform(waveform)
            r = F.normalize(self.wavlm(input_values=x).embeddings, dim=-1)
            hubert_out = self.hubert(input_values=x, output_hidden_states=True)
            h = F.normalize(hubert_out.last_hidden_state.mean(dim=1), dim=-1)
            semantic = F.normalize(hubert_out.hidden_states[self.cfg.semantic_layer].mean(dim=1), dim=-1)
        return {"wavlm": r, "hubert": h, "semantic": semantic}


def cosine(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return F.cosine_similarity(a, b, dim=-1).mean()


def precompute_representations(
    selection: pd.DataFrame,
    cfg: ExperimentConfig,
    models: RepresentationModels,
    device: torch.device,
) -> Dict[str, Dict[str, torch.Tensor]]:
    cache = {}
    for row in selection.to_dict("records"):
        cache[row["path"]] = models.extract(load_audio(row["path"], cfg, device))
    return cache


def audit_audio_levels(selection: pd.DataFrame, cfg: ExperimentConfig, device: torch.device) -> pd.DataFrame:
    rows = []
    for row in selection.to_dict("records"):
        raw_np, raw_sr = sf.read(row["path"], dtype="float32", always_2d=True)
        raw_mono = raw_np.mean(axis=1)
        processed = load_audio(row["path"], cfg, device).squeeze().detach().cpu().numpy()
        raw_rms = float(np.sqrt(np.mean(raw_mono ** 2) + 1e-12))
        processed_rms = float(np.sqrt(np.mean(processed ** 2) + 1e-12))
        rows.append({
            "path": row["path"], "speaker_id": row["speaker_id"],
            "utterance_id": row["utterance_id"], "split": row["split"], "role": row["role"],
            "original_sample_rate_hz": raw_sr, "original_channels": raw_np.shape[1],
            "raw_rms": raw_rms, "raw_rms_dbfs": float(20 * np.log10(raw_rms + 1e-12)),
            "raw_peak": float(np.max(np.abs(raw_mono))),
            "processed_rms": processed_rms, "processed_rms_dbfs": float(20 * np.log10(processed_rms + 1e-12)),
            "processed_peak": float(np.max(np.abs(processed))),
        })
    return pd.DataFrame(rows)


def speaker_centroids(
    selection: pd.DataFrame,
    cache: Dict[str, Dict[str, torch.Tensor]],
    split: str,
    role: Optional[str] = None,
) -> Dict[str, Dict[str, torch.Tensor]]:
    subset = selection[selection.split == split]
    if role is not None:
        subset = subset[subset.role == role]
    result = {}
    for spk, group in subset.groupby("speaker_id"):
        result[spk] = {
            key: F.normalize(torch.stack([cache[p][key] for p in group.path]).mean(dim=0), dim=-1)
            for key in ("wavlm", "hubert", "semantic")
        }
    return result


def choose_hard_negative(
    reference: torch.Tensor,
    candidate_centroids: Dict[str, Dict[str, torch.Tensor]],
) -> Tuple[str, Dict[str, torch.Tensor]]:
    scores = {spk: float(cosine(reference, values["wavlm"]).detach().cpu()) for spk, values in candidate_centroids.items()}
    chosen = max(scores, key=scores.get)
    return chosen, candidate_centroids[chosen]


def eer_threshold(genuine: Sequence[float], impostor: Sequence[float]) -> Tuple[float, float]:
    genuine = np.asarray(genuine, dtype=float)
    impostor = np.asarray(impostor, dtype=float)
    if len(genuine) == 0 or len(impostor) == 0:
        raise ValueError('Calibration requires both genuine and impostor scores.')
    # When calibration classes are separated, every point inside the gap has
    # zero empirical error. Use the midpoint, not the minimum genuine score.
    if np.max(impostor) < np.min(genuine):
        return float((np.max(impostor) + np.min(genuine)) / 2.0), 0.0
    labels = np.r_[np.ones(len(genuine)), np.zeros(len(impostor))]
    scores = np.r_[genuine, impostor]
    fpr, tpr, thresholds = roc_curve(labels, scores)
    fnr = 1.0 - tpr
    idx = int(np.nanargmin(np.abs(fpr - fnr)))
    return float(thresholds[idx]), float((fpr[idx] + fnr[idx]) / 2.0)




In [ ]:
def calibrate_asv(
    selection: pd.DataFrame,
    cache: Dict[str, Dict[str, torch.Tensor]],
    cfg: ExperimentConfig,
) -> Tuple[float, pd.DataFrame, float]:
    # Kalibrasi harus meniru scoring test: centroid dari dua enrollment
    # dibandingkan dengan satu probe yang tidak ikut membentuk centroid.
    calib = selection[selection.split == "calibration"]
    enrollment_size = cfg.enrollment_utterances_per_speaker
    genuine_rows, impostor_candidates, centroid_rows = [], [], []
    probe_rows = calib[["speaker_id", "path", "utterance_id"]].to_dict("records")
    for spk, group in calib.groupby("speaker_id"):
        paths = sorted(group.path.tolist())
        if len(paths) <= enrollment_size:
            raise ValueError(f"Calibration speaker {spk} needs more than {enrollment_size} utterances.")
        for enrollment_paths in combinations(paths, enrollment_size):
            enrollment_set = set(enrollment_paths)
            centroid = F.normalize(
                torch.stack([cache[path]["wavlm"] for path in enrollment_paths]).mean(dim=0),
                dim=-1,
            )
            centroid_rows.append({
                "speaker_id": spk, "enrollment_paths": enrollment_paths, "centroid": centroid,
            })
            for probe_path in paths:
                if probe_path in enrollment_set:
                    continue
                genuine_rows.append({
                    "label": "genuine", "speaker_a": spk, "speaker_b": spk,
                    "enrollment_paths": "|".join(enrollment_paths), "probe_path": probe_path,
                    "score": float(cosine(centroid, cache[probe_path]["wavlm"]).cpu()),
                    "protocol": f"{enrollment_size}-utterance centroid-to-held-out-probe",
                })
    for enrollment in centroid_rows:
        for probe in probe_rows:
            if probe["speaker_id"] == enrollment["speaker_id"]:
                continue
            impostor_candidates.append({
                "label": "impostor", "speaker_a": enrollment["speaker_id"],
                "speaker_b": probe["speaker_id"],
                "enrollment_paths": "|".join(enrollment["enrollment_paths"]),
                "probe_path": probe["path"],
                "score": float(cosine(enrollment["centroid"], cache[probe["path"]]["wavlm"]).cpu()),
                "protocol": f"{enrollment_size}-utterance centroid-to-held-out-probe",
            })
    rng = random.Random(cfg.selection_seed)
    rng.shuffle(impostor_candidates)
    impostor_rows = impostor_candidates[: len(genuine_rows)]
    calibration = pd.DataFrame(genuine_rows + impostor_rows)
    threshold, eer = eer_threshold(
        calibration.loc[calibration.label == "genuine", "score"],
        calibration.loc[calibration.label == "impostor", "score"],
    )
    return threshold, calibration, eer


def precompute_calibration_wavlm(
    selection: pd.DataFrame, cfg: ExperimentConfig, device: torch.device,
) -> Dict[str, Dict[str, torch.Tensor]]:
    """Extract only the 100 calibration embeddings; HuBERT and PGD are skipped."""
    model = WavLMForXVector.from_pretrained(cfg.wavlm_checkpoint).to(device).eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    cache = {}
    subset = selection[selection.split == "calibration"]
    for row in tqdm(subset.to_dict("records"), desc="Calibration WavLM embeddings"):
        waveform = normalize_waveform(load_audio(row["path"], cfg, device))
        with torch.inference_mode():
            embedding = F.normalize(model(input_values=waveform).embeddings, dim=-1)
        cache[row["path"]] = {"wavlm": embedding}
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return cache


class SquimQuality:
    def __init__(self):
        from torchaudio.pipelines import SQUIM_SUBJECTIVE
        self.model = SQUIM_SUBJECTIVE.get_model().cpu().eval()

    def __call__(self, degraded: torch.Tensor, reference: torch.Tensor) -> float:
        with torch.inference_mode():
            score = self.model(degraded.detach().cpu(), reference.detach().cpu())
        return float(score.squeeze().item())


def add_noise_at_snr(x: torch.Tensor, snr_db: float, seed: int) -> torch.Tensor:
    generator = torch.Generator(device=x.device).manual_seed(seed)
    noise = torch.randn(x.shape, generator=generator, device=x.device, dtype=x.dtype)
    signal_rms = torch.sqrt(torch.mean(x.square()) + 1e-12)
    noise_rms = torch.sqrt(torch.mean(noise.square()) + 1e-12)
    scaled = noise * (signal_rms / (10 ** (snr_db / 20.0) * noise_rms))
    return torch.clamp(x + scaled, -1.0, 1.0)


def calibrate_quality(
    selection: pd.DataFrame,
    cfg: ExperimentConfig,
    quality: SquimQuality,
    device: torch.device,
) -> Tuple[float, pd.DataFrame, float, float, float]:
    rows = []
    calib = selection[selection.split == "calibration"]
    for idx, row in enumerate(calib.to_dict("records")):
        clean = load_audio(row["path"], cfg, device)
        degraded = add_noise_at_snr(clean, cfg.quality_calibration_snr_db, cfg.selection_seed + idx)
        clean_np = clean.squeeze().detach().cpu().numpy()
        degraded_np = degraded.squeeze().detach().cpu().numpy()
        rows.append({"label": "clean", "utterance_id": row["utterance_id"], "score": quality(clean, clean), "stoi": 1.0})
        rows.append({"label": f"noise_{cfg.quality_calibration_snr_db:g}dB", "utterance_id": row["utterance_id"], "score": quality(degraded, clean), "stoi": float(stoi(clean_np, degraded_np, cfg.sample_rate, extended=False))})
    table = pd.DataFrame(rows)
    threshold, eer = eer_threshold(
        table.loc[table.label == "clean", "score"],
        table.loc[table.label != "clean", "score"],
    )
    utility_threshold, utility_eer = eer_threshold(
        table.loc[table.label == "clean", "stoi"],
        table.loc[table.label != "clean", "stoi"],
    )
    return threshold, table, eer, utility_threshold, utility_eer




In [ ]:
class PGDProtector:
    def __init__(self, cfg: ExperimentConfig, models: RepresentationModels, device: torch.device, routing_threshold: float):
        self.cfg, self.models, self.device = cfg, models, device
        self.routing_threshold = routing_threshold
        self.window = torch.hann_window(cfg.win_length, device=device)
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=cfg.sample_rate, n_fft=cfg.n_fft, win_length=cfg.win_length,
            hop_length=cfg.hop_length, f_min=cfg.mel_fmin, f_max=cfg.mel_fmax,
            n_mels=cfg.n_mels, center=cfg.stft_center, normalized=cfg.stft_normalized,
            mel_scale=cfg.mel_scale, norm=cfg.mel_norm, power=cfg.mel_power,
        ).to(device)
        mel_centers = librosa.mel_frequencies(
            n_mels=cfg.n_mels, fmin=cfg.mel_fmin, fmax=cfg.mel_fmax,
            htk=cfg.mel_scale == "htk",
        )
        self.mel_centers_hz = mel_centers
        self.formant_bins = cfg.formant_mel_bins
        self.freqs = torch.fft.rfftfreq(cfg.n_fft, d=1.0 / cfg.sample_rate).to(device)

    def stft(self, x: torch.Tensor) -> torch.Tensor:
        return torch.stft(
            x, n_fft=self.cfg.n_fft, hop_length=self.cfg.hop_length,
            win_length=self.cfg.win_length, window=self.window,
            center=self.cfg.stft_center, normalized=self.cfg.stft_normalized,
            onesided=True, return_complex=True,
        )

    def istft(self, x: torch.Tensor, length: int) -> torch.Tensor:
        return torch.istft(
            x, n_fft=self.cfg.n_fft, hop_length=self.cfg.hop_length,
            win_length=self.cfg.win_length, window=self.window,
            center=self.cfg.stft_center, normalized=self.cfg.stft_normalized,
            onesided=True, length=length,
        )

    def filter_gradient(self, gradient: torch.Tensor, phase: int) -> torch.Tensor:
        spectrum = self.stft(gradient)
        scales = torch.ones_like(self.freqs)
        if phase == 1:
            # Paper Section III-D: bins >= 80 (2.5 kHz) are scaled by 0.05.
            scales[self.freqs >= self.cfg.phase1_high_cut_hz] = self.cfg.phase1_high_scale
        else:
            # Paper Section III-D: <45 unchanged, 45..120 -> 0.01, >120 -> 0.15.
            middle = (self.freqs >= self.cfg.ushape_low_end_hz) & (self.freqs <= self.cfg.ushape_mid_end_hz)
            high = self.freqs > self.cfg.ushape_mid_end_hz
            scales[middle] = self.cfg.ushape_mid_scale
            scales[high] = self.cfg.ushape_high_scale
        spectrum = spectrum * scales.view(1, -1, 1)
        return self.istft(spectrum, gradient.shape[-1])

    def filter_high_frequency_gradient(self, gradient: torch.Tensor) -> torch.Tensor:
        """Keep only 3.75-8 kHz at 16 kHz; this is high-frequency, not ultrasonic."""
        spectrum = self.stft(gradient)
        keep = (self.freqs >= self.cfg.high_frequency_start_hz).to(spectrum.dtype)
        spectrum = spectrum * keep.view(1, -1, 1)
        return self.istft(spectrum, gradient.shape[-1])

    def protect(
        self,
        mode: str,
        clean: torch.Tensor,
        reference: Dict[str, torch.Tensor],
        opposite: Dict[str, torch.Tensor],
        seed: int,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        if mode not in {"static", "static_ushape", "dynamic_no_ushape", "high_frequency", "proposed"}:
            raise ValueError(f"Unknown PGD mode: {mode}")
        seed_everything(seed)
        clean_norm = normalize_waveform(clean)
        mel_ref = torch.log(self.mel(clean_norm) + 1e-6).detach()
        f_ref = self.models.extract(clean, with_grad=False)["semantic"]
        envelope = clean.abs() / (clean.abs().amax(dim=-1, keepdim=True) + 1e-7)
        dynamic_bound = self.cfg.epsilon_max * envelope + self.cfg.baseline_noise
        delta = ((torch.rand_like(clean) * 2.0 - 1.0) * (self.cfg.baseline_noise * 0.5)).requires_grad_(True)
        phase2_steps = 0
        phase_switches = 0
        previous_phase = None
        initial_score = float(cosine(self.models.extract(clean)["wavlm"], reference["wavlm"]).cpu())

        for _ in range(self.cfg.num_iterations):
            adversarial = torch.clamp(clean + delta, -1.0, 1.0)
            reps = self.models.extract(adversarial, with_grad=True)
            s_asv = cosine(reps["wavlm"], reference["wavlm"])

            if mode == "static_ushape":
                phase, weights = 1, self.cfg.static_weights
            elif mode in {"static", "high_frequency"}:
                phase, weights = 0, self.cfg.static_weights
            elif float(s_asv.detach().cpu()) > self.routing_threshold:
                phase, weights = 1, self.cfg.phase1_weights
            else:
                phase, weights = 2, self.cfg.phase2_weights
                phase2_steps += 1
            if previous_phase is not None and phase != previous_phase:
                phase_switches += 1
            previous_phase = phase

            loss_identity = cosine(reps["wavlm"], reference["wavlm"]) - cosine(reps["wavlm"], opposite["wavlm"])
            loss_hubert = cosine(reps["hubert"], reference["hubert"]) - cosine(reps["hubert"], opposite["hubert"])
            mel_adv = torch.log(self.mel(normalize_waveform(adversarial)) + 1e-6)
            loss_formant = cosine(
                mel_adv[:, : self.formant_bins].flatten(1),
                mel_ref[:, : self.formant_bins].flatten(1),
            )
            # F is explicitly HuBERT hidden layer `semantic_layer`; minimizing
            # 1-cosine preserves its content representation.
            loss_semantic = 1.0 - cosine(reps["semantic"], f_ref)
            loss = (
                weights["identity"] * loss_identity
                + weights["hubert_identity"] * loss_hubert
                + weights["formant"] * loss_formant
                + weights["semantic"] * loss_semantic
            )
            loss.backward()

            with torch.no_grad():
                # Equations (12)-(14): filter the raw loss gradient in STFT,
                # return with ISTFT, then clip its L-infinity norm. This keeps
                # both the declared band scales and the gradient information.
                raw_gradient = delta.grad
                if mode in {"proposed", "static_ushape"}:
                    direction = self.filter_gradient(raw_gradient, phase)
                    direction = direction / direction.abs().amax(dim=-1, keepdim=True).clamp_min(1e-12)
                elif mode == "high_frequency":
                    direction = self.filter_high_frequency_gradient(raw_gradient)
                    direction = direction / direction.abs().amax(dim=-1, keepdim=True).clamp_min(1e-12)
                else:
                    direction = raw_gradient.sign()
                delta -= self.cfg.step_size * direction
                delta.copy_(torch.maximum(torch.minimum(delta, dynamic_bound), -dynamic_bound))
                delta.grad.zero_()

        protected = torch.clamp(clean + delta.detach(), -1.0, 1.0)
        final_score = float(cosine(self.models.extract(protected)["wavlm"], reference["wavlm"]).cpu())
        return protected, {
            "initial_routing_s_asv": initial_score,
            "final_s_asv": final_score,
            "phase2_fraction": phase2_steps / self.cfg.num_iterations,
            "phase_switches": phase_switches,
        }


def calculate_snr(clean: np.ndarray, test: np.ndarray) -> float:
    signal = np.mean(clean ** 2)
    noise = np.mean((test - clean) ** 2)
    return float(10.0 * np.log10((signal + 1e-12) / (noise + 1e-12)))


def perturbation_band_energy(clean: np.ndarray, test: np.ndarray, cfg: ExperimentConfig) -> Dict[str, float]:
    perturbation = test - clean
    power = np.abs(np.fft.rfft(perturbation)) ** 2
    frequencies = np.fft.rfftfreq(len(perturbation), d=1.0 / cfg.sample_rate)
    total = float(power.sum()) + 1e-20
    masks = {
        "band_energy_low_fraction": frequencies < cfg.ushape_low_end_hz,
        "band_energy_mid_fraction": (frequencies >= cfg.ushape_low_end_hz) & (frequencies <= cfg.ushape_mid_end_hz),
        "band_energy_high_fraction": frequencies > cfg.ushape_mid_end_hz,
    }
    return {name: float(power[mask].sum() / total) for name, mask in masks.items()}


def safe_pesq(clean: np.ndarray, test: np.ndarray, sr: int) -> float:
    try:
        return float(pesq(sr, clean, test, "wb"))
    except Exception:
        return float("nan")


def calculate_ssim(clean: np.ndarray, test: np.ndarray, sr: int) -> float:
    kwargs = dict(sr=sr, n_fft=512, hop_length=160, n_mels=80, fmax=sr / 2)
    a = librosa.power_to_db(librosa.feature.melspectrogram(y=clean, **kwargs), ref=np.max)
    b = librosa.power_to_db(librosa.feature.melspectrogram(y=test, **kwargs), ref=np.max)
    data_range = max(float(a.max() - a.min()), 1e-8)
    return float(structural_similarity(a, b, data_range=data_range))




In [ ]:
def calculate_dtw_and_mcd(clean: np.ndarray, test: np.ndarray, sr: int) -> Tuple[float, float]:
    # Natural-log Mel cepstra with orthonormal DCT; coefficients 1..23.
    mel_kwargs = dict(sr=sr, n_fft=512, hop_length=160, n_mels=40, fmax=sr / 2, power=1.0)
    logmel_clean = np.log(librosa.feature.melspectrogram(y=clean, **mel_kwargs) + 1e-10)
    logmel_test = np.log(librosa.feature.melspectrogram(y=test, **mel_kwargs) + 1e-10)
    a = dct(logmel_clean, type=2, axis=0, norm='ortho')[1:24]
    b = dct(logmel_test, type=2, axis=0, norm='ortho')[1:24]
    cost, path = librosa.sequence.dtw(X=a, Y=b, metric="euclidean")
    path = path[::-1]
    differences = a[:, path[:, 0]] - b[:, path[:, 1]]
    frame_distances = np.sqrt(np.sum(differences ** 2, axis=0))
    mcd = (10.0 * np.sqrt(2.0) / np.log(10.0)) * np.mean(frame_distances)
    dtw = float(cost[-1, -1] / max(len(path), 1))
    return dtw, float(mcd)


def read_transcript(audio_path: str, cfg: ExperimentConfig) -> Optional[str]:
    audio = Path(audio_path)
    if cfg.dataset_name.lower() == "vctk":
        base = re.sub(r"_mic[12]$", "", audio.stem)
        candidates = [audio.with_name(base + ".txt")]
        parts = list(audio.parts)
        for index, part in enumerate(parts):
            if part.lower().startswith("wav48") or part.lower() == "wav16":
                candidates.append(Path(*parts[:index], "txt", audio.parent.name, base + ".txt"))
                break
    else:
        candidates = [audio.with_suffix(".normalized.txt")]
    for txt in candidates:
        if txt.exists():
            return txt.read_text(encoding="utf-8").strip()
    return None


def normalize_text(text: str) -> str:
    return re.sub(r"[^A-Z' ]+", " ", text.upper()).strip()


class ASREvaluator:
    def __init__(self, checkpoint: str, device: torch.device):
        self.processor = Wav2Vec2Processor.from_pretrained(checkpoint)
        self.model = Wav2Vec2ForCTC.from_pretrained(checkpoint).to(device).eval()
        self.device = device

    def transcribe(self, waveform: torch.Tensor, sr: int) -> str:
        values = self.processor(waveform.squeeze().detach().cpu().numpy(), sampling_rate=sr, return_tensors="pt").input_values.to(self.device)
        with torch.inference_mode():
            ids = self.model(values).logits.argmax(dim=-1)
        return normalize_text(self.processor.batch_decode(ids)[0])


def word_error_rate(reference: Optional[str], hypothesis: Optional[str]) -> float:
    if not reference or hypothesis is None:
        return float("nan")
    from jiwer import wer
    return float(wer(normalize_text(reference), normalize_text(hypothesis)))


def evaluate_candidate(
    clean: torch.Tensor,
    candidate: torch.Tensor,
    enrollment: Dict[str, torch.Tensor],
    models: RepresentationModels,
    quality: SquimQuality,
    asr: Optional[ASREvaluator],
    transcript: Optional[str],
    cfg: ExperimentConfig,
    asv_threshold: float,
    calibrated_asv_threshold: float,
    quality_threshold: float,
    utility_threshold: float,
) -> Dict[str, float]:
    clean_np = clean.squeeze().detach().cpu().numpy()
    test_np = candidate.squeeze().detach().cpu().numpy()
    s_asv = float(cosine(models.extract(candidate)["wavlm"], enrollment["wavlm"]).cpu())
    squim_mos = quality(candidate, clean)
    dtw, mcd = calculate_dtw_and_mcd(clean_np, test_np, cfg.sample_rate)
    hypothesis = asr.transcribe(candidate, cfg.sample_rate) if asr is not None and transcript else None
    stoi_score = float(stoi(clean_np, test_np, cfg.sample_rate, extended=False))
    result = {
        "s_asv": s_asv,
        # Endpoint utama memakai threshold EER dari calibration speakers.
        "asv_accept": int(s_asv >= calibrated_asv_threshold),
        "asv_obfuscated": int(s_asv < calibrated_asv_threshold),
        # Threshold 0.25 dipertahankan hanya sebagai audit legacy, bukan ASV EER.
        "asv_accept_fixed_paper": int(s_asv >= asv_threshold),
        "asv_obfuscated_fixed_paper": int(s_asv < asv_threshold),
        "asv_obfuscated_calibrated_eer": int(s_asv < calibrated_asv_threshold),
        "squim_mos": squim_mos,
        "quality_degraded": int(squim_mos < quality_threshold),
        "snr_db": calculate_snr(clean_np, test_np),
        "pesq": safe_pesq(clean_np, test_np, cfg.sample_rate),
        "stoi": stoi_score,
        "quality_preserved": int(stoi_score >= utility_threshold),
        "ssim": calculate_ssim(clean_np, test_np, cfg.sample_rate),
        "mcd_db": mcd,
        "dtw_cost": dtw,
        "wer": word_error_rate(transcript, hypothesis),
        **perturbation_band_energy(clean_np, test_np, cfg),
    }
    # Exact sample-level SQUIM-proxy definition; this is not called NISQA.
    result["dsr_success"] = int(result["asv_obfuscated"] and result["quality_degraded"])
    # Primary privacy-utility criterion: the source must remain intelligible.
    result["privacy_utility_success"] = int(result["asv_obfuscated"] and result["quality_preserved"])
    return result


def package_versions() -> Dict[str, str]:
    import importlib.metadata
    versions = {}
    for name in ("torch", "torchaudio", "transformers", "librosa", "pesq", "pystoi", "scipy", "scikit-learn"):
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = "not-installed"
    return versions




In [ ]:
# =========================
# KONFIGURASI PENGGUNA
# =========================
TARGET_DATASET = "VCTK"  # ganti menjadi "VCTK" untuk run kedua

# None = cari hasil lama secara otomatis di /kaggle/input.
# Dapat diisi folder hasil lama atau file ZIP bila pencarian otomatis tidak sesuai.
OLD_RESULT_SOURCE = "./vctk"

# True mempertahankan WER untuk konsistensi dengan hasil lama, tetapi menambah waktu.
RUN_ASR_WER = True

DATASET_GLOBS = {
    "LibriTTS": "./datasets/libri-tts-train-clean-360-part-2/**/*.wav",
    "VCTK": "./datasets/english-multispeaker-corpus-for-voice-cloning/VCTK-Corpus/VCTK-Corpus/wav48/**/*.wav",
}

METHOD_NEW = "Static PGD + U-Shape"
METHOD_STATIC = "Static PGD"
METHOD_DYNAMIC_NO_FILTER = "Dynamic PGD (No U-Shape)"
METHOD_DYNAMIC_FILTER = "Proposed (Dynamic + U-Shape)"

import glob
import shutil
import zipfile


def _read_manifest(directory: Path) -> Optional[Dict]:
    path = directory / "experiment_manifest.json"
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return None


def _valid_old_result(directory: Path, dataset_name: str) -> bool:
    manifest = _read_manifest(directory)
    metrics_path = directory / "per_sample_metrics.csv"
    selection_path = directory / "sample_selection_manifest.csv"
    if manifest is None or not metrics_path.is_file() or not selection_path.is_file():
        return False
    if str(manifest.get("config", {}).get("dataset_name", "")).casefold() != dataset_name.casefold():
        return False
    try:
        selection = pd.read_csv(selection_path, usecols=["speaker_id", "utterance_id", "split", "role"])
        metrics = pd.read_csv(metrics_path, usecols=["sample_id", "run_seed", "method"])
    except Exception:
        return False
    expected_methods = {
        "Undefended", "Random Noise", "Static PGD",
        "Dynamic PGD (No U-Shape)",
        "High-Frequency PGD (3.75-8 kHz)",
        "Proposed (Dynamic + U-Shape)",
    }
    return (
        len(selection) == 500
        and selection.speaker_id.nunique() == 60
        and len(metrics) == 5760
        and set(metrics.method.unique()) == expected_methods
    )


def resolve_old_result(dataset_name: str, explicit_source: Optional[str]) -> Path:
    if explicit_source:
        source = Path(explicit_source)
        if source.is_file() and source.suffix.lower() == ".zip":
            extracted = Path("/kaggle/working") / f"_old_{dataset_name.lower()}_extracted"
            extracted.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(source) as archive:
                archive.extractall(extracted)
            search_root = extracted
        elif source.is_dir():
            search_root = source
        else:
            raise FileNotFoundError(f"OLD_RESULT_SOURCE tidak ditemukan: {source}")
        candidates = [search_root] + [p.parent for p in search_root.rglob("experiment_manifest.json")]
    else:
        input_root = Path("/kaggle/input")
        candidates = [p.parent for p in input_root.rglob("experiment_manifest.json")]

    valid = sorted(
        {candidate.resolve() for candidate in candidates if _valid_old_result(candidate, dataset_name)},
        key=lambda path: (int((path / "asv_centroid_recalibration_status.csv").is_file()), len(path.parts), str(path)),
        reverse=True,
    )
    if not valid:
        raise FileNotFoundError(
            f"Hasil final {dataset_name} tidak ditemukan. Pasang ZIP/folder hasil lama "
            "sebagai Kaggle Input atau isi OLD_RESULT_SOURCE secara eksplisit."
        )
    return valid[0]


def resolve_audio_paths(saved_selection: pd.DataFrame, dataset_name: str, dataset_glob: str) -> pd.DataFrame:
    files = sorted(glob.glob(dataset_glob, recursive=True))
    if not files:
        raise FileNotFoundError(f"Dataset WAV tidak ditemukan dengan pola: {dataset_glob}")

    index: Dict[Tuple[str, str], List[str]] = {}
    for path in files:
        path_obj = Path(path)
        spk = path_obj.parent.name if dataset_name == "VCTK" else path_obj.parent.parent.name
        source_id = re.sub(r"_mic[12]$", "", path_obj.stem) if dataset_name == "VCTK" else path_obj.stem
        index.setdefault((str(spk), source_id), []).append(path)

    resolved = saved_selection.copy()
    resolved_paths = []
    for row in resolved.to_dict("records"):
        source_id = str(row.get("source_utterance_id", row["utterance_id"]))
        if dataset_name == "VCTK":
            source_id = re.sub(r"_mic[12]$", "", source_id)
        candidates = index.get((str(row["speaker_id"]), source_id), [])
        if dataset_name == "VCTK" and len(candidates) > 1:
            mic1 = [path for path in candidates if Path(path).stem.endswith("_mic1")]
            candidates = mic1 or candidates
        if not candidates:
            raise FileNotFoundError(
                f"Audio manifest tidak ditemukan dalam dataset: speaker={row['speaker_id']}, "
                f"utterance={source_id}"
            )
        resolved_paths.append(sorted(candidates)[0])

    resolved["path"] = resolved_paths
    if len(resolved) != 500 or resolved.speaker_id.nunique() != 60:
        raise RuntimeError("Manifest hasil lama tidak lagi berisi tepat 500 utterances dari 60 speakers.")
    if resolved.path.duplicated().any():
        raise RuntimeError("Resolusi dataset menghasilkan path audio duplikat.")
    return resolved


if TARGET_DATASET not in DATASET_GLOBS:
    raise ValueError("TARGET_DATASET harus 'LibriTTS' atau 'VCTK'.")

old_result_dir = resolve_old_result(TARGET_DATASET, OLD_RESULT_SOURCE)
old_manifest = _read_manifest(old_result_dir)
old_results = pd.read_csv(old_result_dir / "per_sample_metrics.csv")
saved_selection = pd.read_csv(old_result_dir / "sample_selection_manifest.csv")

config_values = {
    key: value for key, value in old_manifest["config"].items()
    if key in ExperimentConfig.__dataclass_fields__
}
config_values.update({
    "dataset_name": TARGET_DATASET,
    "dataset_glob": DATASET_GLOBS[TARGET_DATASET],
    "output_dir": f"/kaggle/working/factorial_ablation_{TARGET_DATASET.lower()}",
    "run_asr_wer": RUN_ASR_WER,
    "checkpoint_every": 10,
})
if "run_seeds" in config_values:
    config_values["run_seeds"] = tuple(config_values["run_seeds"])
cfg = ExperimentConfig(**config_values)
cfg.validate()

selection = resolve_audio_paths(saved_selection, TARGET_DATASET, cfg.dataset_glob)
output_dir = Path(cfg.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
selection.to_csv(output_dir / "resolved_sample_selection_manifest.csv", index=False)

calibration_info = old_manifest["calibration"]
legacy_asv_threshold = float(
    calibration_info.get("legacy_fixed_score_threshold_audit_only",
                         calibration_info.get("paper_asv_evaluation_threshold", 0.25))
)
calibrated_asv_threshold = float(
    calibration_info.get("calibrated_asv_eer_threshold",
                         calibration_info.get("primary_asv_evaluation_threshold",
                                              calibration_info.get("calibration_eer_threshold_for_audit_only")))
)
quality_threshold = float(calibration_info["quality_threshold"])
utility_threshold = float(calibration_info["utility_threshold"])


In [ ]:
%%capture
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("GPU Kaggle harus diaktifkan untuk eksperimen 1000 iterasi.")

models = RepresentationModels(cfg, device)
cache = precompute_representations(selection, cfg, models, device)
calibration_centroids = speaker_centroids(selection, cache, "calibration")
enrollment_centroids = speaker_centroids(selection, cache, "test", role="enrollment")
protector = PGDProtector(cfg, models, device, cfg.routing_threshold)
quality = SquimQuality()
asr = ASREvaluator(cfg.asr_checkpoint, device) if cfg.run_asr_wer else None

probes = selection[(selection.split == "test") & (selection.role == "probe")].reset_index(drop=True)
if len(probes) != 320 or probes.speaker_id.nunique() != 40:
    raise RuntimeError("Eksperimen tambahan wajib menggunakan 320 probe dari 40 test speakers.")

checkpoint_path = output_dir / "static_ushape_checkpoint.csv"
result_rows = pd.read_csv(checkpoint_path).to_dict("records") if checkpoint_path.is_file() else []
completed = {(str(row["sample_id"]), int(row["run_seed"])) for row in result_rows}

for run_seed in cfg.run_seeds:
    for sample_index, row in enumerate(probes.to_dict("records")):
        key = (str(row["utterance_id"]), int(run_seed))
        if key in completed:
            continue

        clean = load_audio(row["path"], cfg, device)
        enrollment = enrollment_centroids[row["speaker_id"]]
        opposite_speaker, opposite = choose_hard_negative(enrollment["wavlm"], calibration_centroids)
        sample_seed = int(run_seed) * 100_000 + sample_index

        protected, diagnostics = protector.protect(
            "static_ushape", clean, enrollment, opposite, sample_seed
        )
        transcript = read_transcript(row["path"], cfg)
        metrics = evaluate_candidate(
            clean, protected, enrollment, models, quality, asr, transcript, cfg,
            legacy_asv_threshold, calibrated_asv_threshold,
            quality_threshold, utility_threshold,
        )
        result_rows.append({
            "sample_id": row["utterance_id"],
            "speaker_id": row["speaker_id"],
            "path": row["path"],
            "run_seed": int(run_seed),
            "method": METHOD_NEW,
            "opposite_speaker_id": opposite_speaker,
            "phase2_fraction": 0.0,
            "phase_switches": 0,
            "routing_s_asv_initial": diagnostics["initial_routing_s_asv"],
            "routing_s_asv_final": diagnostics["final_s_asv"],
            "asv_threshold": calibrated_asv_threshold,
            "quality_threshold": quality_threshold,
            "utility_threshold": utility_threshold,
            "asv_threshold_calibrated_eer": calibrated_asv_threshold,
            "asv_threshold_fixed_paper": legacy_asv_threshold,
            **metrics,
        })
        completed.add(key)

        if run_seed == cfg.run_seeds[0] and sample_index == 0:
            sf.write(
                output_dir / "Static_PGD_plus_U_Shape_example.wav",
                protected.squeeze().detach().cpu().numpy(), cfg.sample_rate,
            )

        if len(result_rows) % cfg.checkpoint_every == 0:
            pd.DataFrame(result_rows).to_csv(checkpoint_path, index=False)

new_results = pd.DataFrame(result_rows)
expected_rows = len(probes) * len(cfg.run_seeds)
if len(new_results) != expected_rows:
    raise RuntimeError(f"Metode tambahan menghasilkan {len(new_results)}, seharusnya {expected_rows} records.")
if new_results[["sample_id", "run_seed"]].duplicated().any():
    raise RuntimeError("Checkpoint metode tambahan mengandung pasangan sample-seed duplikat.")

new_results.to_csv(output_dir / "static_ushape_per_sample_metrics.csv", index=False)
combined_results = pd.concat([old_results, new_results], ignore_index=True, sort=False)
combined_results.to_csv(output_dir / "combined_per_sample_metrics.csv", index=False)


In [ ]:
CONTINUOUS_METRICS = {
    "s_asv": "lower",
    "pesq": "higher",
    "stoi": "higher",
    "ssim": "higher",
    "squim_mos": "higher",
    "snr_db": "higher",
    "mcd_db": "lower",
    "dtw_cost": "lower",
}
BINARY_METRICS = {
    "privacy_utility_success": "higher",
    "dsr_success": "higher",
}
FACTORIAL_METHODS = [
    METHOD_STATIC,
    METHOD_NEW,
    METHOD_DYNAMIC_NO_FILTER,
    METHOD_DYNAMIC_FILTER,
]
CONTRASTS = {
    "filter_effect_fixed": (METHOD_NEW, METHOD_STATIC),
    "filter_effect_dynamic": (METHOD_DYNAMIC_FILTER, METHOD_DYNAMIC_NO_FILTER),
    "routing_effect_unfiltered": (METHOD_DYNAMIC_NO_FILTER, METHOD_STATIC),
    "routing_effect_filtered": (METHOD_DYNAMIC_FILTER, METHOD_NEW),
}


def holm_adjust(p_values: Sequence[float]) -> np.ndarray:
    values = np.asarray(p_values, dtype=float)
    order = np.argsort(values)
    adjusted = np.empty(len(values), dtype=float)
    running = 0.0
    total = len(values)
    for rank, index in enumerate(order):
        running = max(running, (total - rank) * values[index])
        adjusted[index] = min(running, 1.0)
    return adjusted


def speaker_level_values(results: pd.DataFrame) -> pd.DataFrame:
    continuous = results.groupby(["speaker_id", "method"], as_index=False)[list(CONTINUOUS_METRICS)].mean()
    rates = results.groupby(["speaker_id", "method"], as_index=False)[list(BINARY_METRICS)].mean()
    return continuous.merge(rates, on=["speaker_id", "method"], how="inner")


def factorial_summary(results: pd.DataFrame) -> pd.DataFrame:
    per_speaker = speaker_level_values(results[results.method.isin(FACTORIAL_METHODS)])
    rows = []
    for method, group in per_speaker.groupby("method"):
        for metric in list(CONTINUOUS_METRICS) + list(BINARY_METRICS):
            scale = 100.0 if metric in BINARY_METRICS else 1.0
            values = group[metric].dropna().to_numpy(float) * scale
            half = (
                float(stats.t.ppf(0.975, len(values) - 1) * stats.sem(values))
                if len(values) > 1 else float("nan")
            )
            rows.append({
                "dataset": TARGET_DATASET,
                "method": method,
                "metric": metric,
                "n_speakers": len(values),
                "mean": float(np.mean(values)),
                "ci95_half_width": half,
            })
    return pd.DataFrame(rows)


def factorial_pairwise_tests(results: pd.DataFrame) -> pd.DataFrame:
    per_speaker = speaker_level_values(results[results.method.isin(FACTORIAL_METHODS)])
    rows = []
    for contrast, (method_a, method_b) in CONTRASTS.items():
        for metric, direction in CONTINUOUS_METRICS.items():
            wide = per_speaker.pivot(index="speaker_id", columns="method", values=metric).dropna()
            if method_a not in wide or method_b not in wide:
                continue
            a, b = wide[method_a], wide[method_b]
            try:
                test = stats.wilcoxon(a, b, zero_method="zsplit", alternative="two-sided")
                statistic, p_value = float(test.statistic), float(test.pvalue)
            except ValueError:
                statistic, p_value = float("nan"), 1.0
            rows.append({
                "dataset": TARGET_DATASET,
                "family": "primary" if metric in {"s_asv", "pesq", "stoi", "ssim", "squim_mos"} else "distortion",
                "contrast": contrast,
                "method_a": method_a,
                "method_b": method_b,
                "metric": metric,
                "preferred_direction": direction,
                "n_speakers": len(wide),
                "mean_a": float(a.mean()),
                "mean_b": float(b.mean()),
                "a_minus_b": float((a - b).mean()),
                "test": "two-sided paired Wilcoxon on speaker means",
                "statistic": statistic,
                "p_value_raw": p_value,
            })

        for metric, direction in BINARY_METRICS.items():
            binary = per_speaker.copy()
            binary["majority"] = (binary[metric] >= 0.5).astype(int)
            wide = binary.pivot(index="speaker_id", columns="method", values="majority").dropna()
            if method_a not in wide or method_b not in wide:
                continue
            a_only = int(((wide[method_a] == 1) & (wide[method_b] == 0)).sum())
            b_only = int(((wide[method_a] == 0) & (wide[method_b] == 1)).sum())
            discordant = a_only + b_only
            p_value = float(stats.binomtest(a_only, discordant, 0.5).pvalue) if discordant else 1.0
            rows.append({
                "dataset": TARGET_DATASET,
                "family": "primary",
                "contrast": contrast,
                "method_a": method_a,
                "method_b": method_b,
                "metric": metric,
                "preferred_direction": direction,
                "n_speakers": len(wide),
                "mean_a": float(per_speaker.loc[per_speaker.method == method_a, metric].mean() * 100),
                "mean_b": float(per_speaker.loc[per_speaker.method == method_b, metric].mean() * 100),
                "a_minus_b": float(
                    (per_speaker.loc[per_speaker.method == method_a, metric].to_numpy()
                     - per_speaker.loc[per_speaker.method == method_b, metric].to_numpy()).mean() * 100
                ),
                "test": "exact paired McNemar on speaker-majority decisions",
                "statistic": a_only - b_only,
                "p_value_raw": p_value,
            })

    table = pd.DataFrame(rows)
    table["p_value_holm"] = np.nan
    for family, indices in table.groupby("family").groups.items():
        table.loc[indices, "p_value_holm"] = holm_adjust(table.loc[indices, "p_value_raw"])
    table["holm_significant_0.05"] = table.p_value_holm < 0.05
    return table


def factorial_interactions(results: pd.DataFrame) -> pd.DataFrame:
    per_speaker = speaker_level_values(results[results.method.isin(FACTORIAL_METHODS)])
    rows = []
    for metric, direction in CONTINUOUS_METRICS.items():
        wide = per_speaker.pivot(index="speaker_id", columns="method", values=metric).dropna()
        required = set(FACTORIAL_METHODS)
        if not required.issubset(wide.columns):
            continue
        filter_dynamic = wide[METHOD_DYNAMIC_FILTER] - wide[METHOD_DYNAMIC_NO_FILTER]
        filter_fixed = wide[METHOD_NEW] - wide[METHOD_STATIC]
        interaction = filter_dynamic - filter_fixed
        try:
            test = stats.wilcoxon(interaction, zero_method="zsplit", alternative="two-sided")
            statistic, p_value = float(test.statistic), float(test.pvalue)
        except ValueError:
            statistic, p_value = float("nan"), 1.0
        rows.append({
            "dataset": TARGET_DATASET,
            "metric": metric,
            "preferred_direction": direction,
            "n_speakers": len(interaction),
            "filter_effect_dynamic": float(filter_dynamic.mean()),
            "filter_effect_fixed": float(filter_fixed.mean()),
            "difference_in_differences": float(interaction.mean()),
            "test": "two-sided paired Wilcoxon of speaker-level difference-in-differences",
            "statistic": statistic,
            "p_value_raw": p_value,
        })
    table = pd.DataFrame(rows)
    table["p_value_holm"] = holm_adjust(table.p_value_raw) if len(table) else []
    table["holm_significant_0.05"] = table.p_value_holm < 0.05
    return table


def plot_factorial(summary: pd.DataFrame, output_path: Path) -> None:
    metrics = ["s_asv", "pesq", "stoi", "ssim", "snr_db", "mcd_db"]
    labels = {
        METHOD_STATIC: "Fixed\nNo filter",
        METHOD_NEW: "Fixed\nU-Shape",
        METHOD_DYNAMIC_NO_FILTER: "Dynamic\nNo filter",
        METHOD_DYNAMIC_FILTER: "Dynamic\nU-Shape",
    }
    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    for ax, metric in zip(axes.flat, metrics):
        subset = summary[summary.metric == metric].set_index("method").reindex(FACTORIAL_METHODS)
        x = np.arange(len(FACTORIAL_METHODS))
        ax.bar(x, subset["mean"], yerr=subset["ci95_half_width"], capsize=3, color=["#777777", "#4C78A8", "#B279A2", "#E45756"])
        ax.set_xticks(x, [labels[method] for method in FACTORIAL_METHODS], fontsize=8)
        ax.set_title(metric)
        ax.grid(axis="y", alpha=0.2)
    fig.suptitle(f"2x2 routing-filtering ablation: {TARGET_DATASET}")
    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)


In [ ]:
%%capture
factorial_methods_present = set(combined_results.method.unique())
missing = set(FACTORIAL_METHODS) - factorial_methods_present
if missing:
    raise RuntimeError(f"Metode faktorial belum lengkap: {sorted(missing)}")

summary = factorial_summary(combined_results)
pairwise = factorial_pairwise_tests(combined_results)
interactions = factorial_interactions(combined_results)

summary.to_csv(output_dir / "factorial_speaker_summary.csv", index=False)
pairwise.to_csv(output_dir / "factorial_pairwise_tests.csv", index=False)
interactions.to_csv(output_dir / "factorial_interaction_tests.csv", index=False)

table = summary.pivot(index="method", columns="metric", values="mean").reindex(FACTORIAL_METHODS)
table.to_csv(output_dir / "factorial_results_table.csv")
plot_factorial(summary, output_dir / "factorial_ablation.png")

run_manifest = {
    "dataset": TARGET_DATASET,
    "new_method": METHOD_NEW,
    "old_result_source": str(old_result_dir),
    "records_new": int(len(new_results)),
    "records_combined": int(len(combined_results)),
    "new_method_speakers": int(new_results.speaker_id.nunique()),
    "new_method_probes": int(new_results.sample_id.nunique()),
    "run_seeds": [int(seed) for seed in cfg.run_seeds],
    "iterations": int(cfg.num_iterations),
    "weights": dict(cfg.static_weights),
    "routing": "disabled; State-A weights fixed for all iterations",
    "gradient_filter": {
        "0_to_2.5_kHz": 1.0,
        "2.5_to_8_kHz": float(cfg.phase1_high_scale),
    },
    "paired_initialization": "sample_seed = run_seed * 100000 + original probe index",
    "thresholds_reused_from_old_result": {
        "asv_calibrated_eer": calibrated_asv_threshold,
        "asv_legacy_audit_only": legacy_asv_threshold,
        "squim": quality_threshold,
        "stoi": utility_threshold,
    },
    "device": str(device),
    "gpu": torch.cuda.get_device_name(0),
}
(output_dir / "static_ushape_experiment_manifest.json").write_text(
    json.dumps(run_manifest, indent=2), encoding="utf-8"
)

validation = {
    "status": "PASS",
    "same_320_probes_as_old_results": bool(
        set(new_results.sample_id.astype(str))
        == set(old_results.loc[old_results.method == METHOD_STATIC, "sample_id"].astype(str))
    ),
    "same_three_seeds": sorted(new_results.run_seed.astype(int).unique().tolist())
        == sorted(int(seed) for seed in cfg.run_seeds),
    "exactly_960_new_records": len(new_results) == 960,
    "exactly_40_test_speakers": new_results.speaker_id.nunique() == 40,
    "no_duplicate_sample_seed": not new_results[["sample_id", "run_seed"]].duplicated().any(),
}
validation["status"] = "PASS" if all(value for key, value in validation.items() if key != "status") else "FAIL"
(output_dir / "validation.json").write_text(json.dumps(validation, indent=2), encoding="utf-8")
if validation["status"] != "PASS":
    raise RuntimeError(f"Validasi eksperimen tambahan gagal: {validation}")

archive_base = f"/kaggle/working/static_ushape_factorial_{TARGET_DATASET.lower()}"
shutil.make_archive(archive_base, "zip", output_dir)


## File keluaran

Setelah `Run All` selesai, unduh ZIP berikut dari panel **Output** Kaggle:

- LibriTTS: `static_ushape_factorial_libritts.zip`
- VCTK: `static_ushape_factorial_vctk.zip`

File terpenting di dalam ZIP:

- `static_ushape_per_sample_metrics.csv`: 960 record metode baru;
- `combined_per_sample_metrics.csv`: hasil lama ditambah metode baru;
- `factorial_speaker_summary.csv`: mean dan 95% CI berbasis 40 speaker;
- `factorial_pairwise_tests.csv`: empat contrast routing/filtering;
- `factorial_interaction_tests.csv`: difference-in-differences faktorial;
- `factorial_ablation.png`: grafik ringkas;
- `validation.json`: pemeriksaan kesamaan probe, seed, dan jumlah record.
